**Metrics used:**
- NDCG@10    → ranking quality (primary metric)
- Precision@10 → fraction of top-10 that are relevant
- Recall@10    → fraction of relevant movies in top-10

**Why NOT accuracy or RMSE?**


- Recommender systems are ranking problems, not regression problems.
- RMSE measures prediction error per rating.
- NDCG measures whether relevant movies appear at the TOP of the list.
- A model can have low RMSE but terrible NDCG — useless in practice.

In [2]:
#importing libraries
import pandas as pd 
import numpy as np
import os 
import pickle
import warnings
from datetime import datetime
from collections import defaultdict

from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate

from tqdm import tqdm

warnings.filterwarnings('ignore')
print("Libraries loaded.")
print(f"Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Libraries loaded.
Run started: 2026-04-04 13:45:08


In [3]:
#Getting the project root path 
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")
MODELS_DIR = os.path.join(os.path.join(BASE_DIR, "models"))
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Base directory: {BASE_DIR}")


Base directory: c:\Projects\Cinemate V2


In [4]:
#Load processed directories
print("Loading preprocessed file...............")


train = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "train.parquet"))
test = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "test.parquet"))
movies_clean = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "movies_clean.parquet"))

with open(os.path.join(PROCESSED_DATA_DIR, "dataset_constants.pkl"), "rb") as f:
    constants = pickle.load(f)

NUM_USERS = constants["NUM_USERS"]
NUM_MOVIES = constants["NUM_MOVIES"]

with open(os.path.join(PROCESSED_DATA_DIR, "user_positive_sets.pkl") , "rb") as f:
    user_positive_sets = pickle.load(f)

with open(os.path.join(PROCESSED_DATA_DIR, "encoders/idx2movie.pkl"), "rb") as f:
    idx2movie = pickle.load(f)

with open(os.path.join(PROCESSED_DATA_DIR, "encoders/idx2user.pkl"), 'rb') as f:
    idx2user = pickle.load(f)

print(f"Train ratings: {len(train)}")
print(f"Test ratings : {len(test)}")
print(f"Num Users : {NUM_USERS}")
print(f"Num Movies : {NUM_MOVIES}")


Loading preprocessed file...............
Train ratings: 25868311
Test ratings : 795563
Num Users : 173134
Num Movies : 27766


### Evaluation metrics

NDCG@K (Normalised Discounted Cumulative Gain):
- Rewards relevant movies appearing HIGH in the ranked list
- A relevant movie at position 1 scores more than at position 10
- Range: 0.0 (worst) to 1.0 (perfect)
- This is the PRIMARY metric

Precision@K:
- Of the top-K recommendations, what fraction did the user like?
- Range: 0.0 to 1.0

Recall@K:
- Of all movies the user liked, what fraction appear in top-K?
- Range: 0.0 to 1.0

In [5]:
def ndcg_at_k(recommended, relevant_sets, k=10):
    """
    recommended  : ordered list of movie_idx (best first)
    relevant_set : set of movie_idx user actually liked in test
    k            : cutoff
    """
    dcg = 0
    for i, movie in enumerate(recommended[:k]):
        if movie in relevant_sets:
            # Position is 1-indexed, log base 2
            dcg +=1.0/np.log2(i+2)
    
    # Ideal DCG — if all top-k were relevant
    ideal_hits = min(k, len(relevant_sets))
    idcg = sum(1.0/np.log2(i+2) for i in range(ideal_hits))

    return dcg/idcg if idcg > 0 else 0.0

def precision_at_k(recommended, relevant_set, k=10):
    hits = len(set(recommended[:k]) & relevant_set)
    return hits/k

def recall_at_k(recommended, relevant_set, k=10):
    if not relevant_set:
        return 0.0
    hits = len(set(recommended[:k]) & relevant_set)
    return hits / len(relevant_set)



In [6]:
def evaluate_recommendations(recomendations_dict, test_relevant_dict, k=10):
    """
    recommendations_dict : {user_idx: [ranked movie_idxs]}
    test_relevant_dict   : {user_idx: set of relevant movie_idxs}
    Returns mean NDCG, Precision, Recall across all users
    """
    ndcg_scores =[]
    precision_scores = []
    recall_scores = []

    for user_idx, recommended in tqdm(recomendations_dict.items(), desc="Evaluating"):
        relevant= test_relevant_dict.get(user_idx, set())
        if not relevant:
            continue

        ndcg_scores.append(ndcg_at_k(recommended, relevant, k))
        precision_scores.append(precision_at_k(recommended, relevant, k))
        recall_scores.append(recall_at_k(recommended, relevant, k))

    return {
        f'NDCG@{k}'     : np.mean(ndcg_scores),
        f'Precision@{k}': np.mean(precision_scores),
        f'Recall@{k}'   : np.mean(recall_scores),
        'n_users_eval'  : len(ndcg_scores)
    }


print("Metric functions defined.")
print("  ndcg_at_k      — ranking quality")
print("  precision_at_k — hit fraction in top-K")
print("  recall_at_k    — coverage of relevant items")

Metric functions defined.
  ndcg_at_k      — ranking quality
  precision_at_k — hit fraction in top-K
  recall_at_k    — coverage of relevant items


In [7]:
# Building test relvant sets
# For each user in test — what movies did they actually like?
# Positive threshold same as preprocessing: >= 3.5

POSITIVE_THRESHOLD = 3.5

test_relevant = (
    test[test['rating']>POSITIVE_THRESHOLD].groupby("user_idx")['movie_idx'].apply(set).to_dict()
)

# Users who have at least one relevant item in test
eval_users = [u for u, rel in test_relevant.items() if len(rel) > 0]

print(f"Test users total              : {test['user_idx'].nunique()}")
print(f"Test users with relevant items: {len(eval_users):,}")
print(f"Avg relevant movies per user  : {np.mean([len(v) for v in test_relevant.values()]):.1f}")


Test users total              : 7835
Test users with relevant items: 7,639
Avg relevant movies per user  : 41.7


### Baseline 1 — Random Recommender

The dumbest possible recommender.
Recommends K random movies the user hasn't seen.

Why include this?
It sets the floor. Any model that can't beat random
is completely useless and should be discarded.
Expected NDCG@10 ≈ 0.01–0.03

In [8]:
all_movie_idxs = list(range(NUM_MOVIES))
N_EVAL_USERS =min(2000, len(eval_users))
eval_samples= eval_users[:N_EVAL_USERS]

random_recs = {}
np.random.seed(42)

for user_idx in tqdm(eval_samples, desc = "Random baseline"):
    seen = user_positive_sets.get(user_idx, set())
    #Sample from unseen movies
    unseen = [m for m in all_movie_idxs if m not in seen]
    random_recs[user_idx]= list(np.random.choice(unseen, size=10, replace=False))

test_relevant_sample = { u: test_relevant[u] for u in eval_samples if u in test_relevant}
random_results= evaluate_recommendations(random_recs, test_relevant_sample, k=10)

print("Random Baseline Results : \n")
for metric ,value in random_results.items():
    if metric != "n_user_eval":
        print(f"  {metric:<15s}: {value:.4f}")
print(f"  Users evaluated: {random_results['n_users_eval']:,}")

Evaluating: 100%|██████████| 2000/2000 [00:00<00:00, 24096.68it/s]

Random Baseline Results : 

  NDCG@10        : 0.0019
  Precision@10   : 0.0018
  Recall@10      : 0.0004
  n_users_eval   : 2000.0000
  Users evaluated: 2,000


In [ ]:
user_positive_sets.get(2, set())

{0: {0,
  108,
  156,
  257,
  351,
  376,
  588,
  1012,
  1040,
  1164,
  1168,
  1177,
  1181,
  1256,
  1481,
  1864,
  1865,
  1933,
  1990,
  2020,
  2237,
  2571,
  2662,
  2808,
  3454,
  4109,
  4168,
  4559,
  4741,
  4751,
  4848,
  4850,
  5785,
  6351,
  6864,
  6922,
  6951,
  6960,
  7161,
  7327,
  7548,
  7756,
  7831,
  8135,
  8143,
  8154,
  9666,
  10125,
  10906,
  11171,
  11323,
  11352},
 1: {0,
  5,
  16,
  20,
  33,
  35,
  46,
  49,
  108,
  139,
  148,
  149,
  163,
  228,
  258,
  263,
  292,
  296,
  314,
  334,
  344,
  351,
  352,
  359,
  362,
  372,
  375,
  429,
  452,
  495,
  522,
  534,
  578,
  579,
  580,
  581,
  582,
  585,
  587,
  589,
  769},
 2: {257,
  314,
  351,
  587,
  893,
  1069,
  2225,
  2758,
  4168,
  5460,
  5828,
  7651,
  7756,
  10125,
  10131,
  10583,
  10785,
  11017,
  12773,
  12902,
  14030,
  14033,
  14472,
  17860,
  18171,
  20705,
  20795},
 3: {46,
  173,
  254,
  314,
  315,
  332,
  522,
  761,
  1117,
  1145,


### Baseline 2 — Popularity Recommender

Recommends the most popular movies globally
(most rated in training set), filtered by what
the user hasn't already seen.

Why include this?
Popularity baseline is surprisingly hard to beat.
Many production recommenders struggle to outperform it
for new users (cold start).

Expected NDCG@10 ≈ 0.05–0.12
If SVD can't beat this — something is wrong with the model.

In [10]:
#Compute popularity: number of ratings per movie in train

movie_popularity = (train.groupby("movie_idx").size().sort_values(ascending=False))
popular_movies= movie_popularity.index.to_list()

popularity_recs = {}

for user_idx in tqdm(eval_samples, desc="Popularity Baseline"):
    seen = user_positive_sets.get(user_idx, set())
    top_k  = [m for m in popular_movies if m not in seen][:10]
    popularity_recs[user_idx] = top_k

popularity_results = evaluate_recommendations(popularity_recs, test_relevant_sample, k=10)

print("Popularity Baseline Results:")
for metric, value in popularity_results.items():
    if metric != 'n_users_eval':
        print(f"  {metric:<15s}: {value:.4f}")
print(f"  Users evaluated: {popularity_results['n_users_eval']:,}")

Evaluating: 100%|██████████| 2000/2000 [00:00<00:00, 20197.74it/s]

Popularity Baseline Results:
  NDCG@10        : 0.0845
  Precision@10   : 0.0749
  Recall@10      : 0.0225
  Users evaluated: 2,000


### Model — SVD (Singular Value Decomposition)


SVD decomposes the user-item rating matrix into:
    R ≈ U × Σ × V^T

Where:
- U : user latent factor matrix  (173,134 × n_factors)
- Σ : diagonal matrix of singular values
- V : item latent factor matrix  (27,766  × n_factors)

Each user and movie is represented as a dense vector
of n_factors dimensions. The dot product of a user
vector and movie vector gives the predicted rating.

Why SVD as baseline and not a simple model?
SVD is the classical, well-understood method.
It's what the Netflix Prize winning team used in 2009.
Beating it with NCF is meaningful — it's not a toy baseline.

We use the Surprise library implementation which handles:
- Bias terms per user and movie
- SGD optimisation
- Regularisation to prevent overfitting

In [11]:
len(train)

25868311

In [12]:
# Surprise needs ratings in original float format
# We use a sample for faster training on this notebook
# Full training happens in train.py with PyTorch

# Sample 5M ratings for surprise — training 25M is very slow
SAMPLE_SIZE = 23_000_000
print(f"Sampling {SAMPLE_SIZE:,} ratings for SVD training...")
print(f"(Full 25M training done in train.py with PyTorch NCF)")
print()

train_sample = train.sample(n=SAMPLE_SIZE, random_state=42).copy()

# Convert float16 rating back to float32 for Surprise
train_sample['rating'] = train_sample['rating'].astype(np.float32)

# Surprise Reader — specify rating scale
reader = Reader(rating_scale=(0.5, 5.0))

# Build Surprise dataset from pandas dataframe
surprise_data = Dataset.load_from_df(train_sample[['user_idx', 'movie_idx', 'rating']],reader)

print(f"Surprise dataset built")
print(f"  Ratings  : {SAMPLE_SIZE:,}")
print(f"  Rating scale: 0.5 – 5.0")


Sampling 23,000,000 ratings for SVD training...
(Full 25M training done in train.py with PyTorch NCF)

Surprise dataset built
  Ratings  : 23,000,000
  Rating scale: 0.5 – 5.0


In [13]:
#train svd
svd_model = SVD(n_factors=150,
                n_epochs =30,
                lr_all= 0.005,
                reg_all = 0.01,
                random_state = 42,
                verbose=True
                )

# Build full trainset and fit
trainset = surprise_data.build_full_trainset()
svd_model.fit(trainset)

print("\nSVD training complete.")

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Processing epoch 20
Processing epoch 21
Processing epoch 22
Processing epoch 23
Processing epoch 24
Processing epoch 25
Processing epoch 26
Processing epoch 27
Processing epoch 28
Processing epoch 29

SVD training complete.


In [14]:
def get_svd_recommendations(model, user_idx,
                             seen_movies, all_movies,
                             k=10):
    """
    Generate top-K recommendations for one user
    using SVD predicted ratings.
    """
    unseen = [m for m in all_movies if m not in seen_movies]

    # Predict rating for each unseen movie
    predictions = [
        (movie, model.predict(user_idx, movie).est)
        for movie in unseen
    ]

    # Sort by predicted rating descending
    predictions.sort(key=lambda x: x[1], reverse=True)

    return [movie for movie, _ in predictions[:k]]


# Generate for eval sample
print(f"Generating SVD recommendations for {N_EVAL_USERS:,} users...")
print("(This takes a few minutes — predicting per unseen movie per user)")
print()

svd_recs = {}

for user_idx in tqdm(eval_samples, desc="SVD recommendations"):
    seen  = user_positive_sets.get(user_idx, set())
    recs  = get_svd_recommendations(
        svd_model, user_idx, seen, all_movie_idxs, k=10
    )
    svd_recs[user_idx] = recs

print("SVD recommendations generated.")

Generating SVD recommendations for 2,000 users...
(This takes a few minutes — predicting per unseen movie per user)



SVD recommendations: 100%|██████████| 2000/2000 [10:18<00:00,  3.23it/s]

SVD recommendations generated.


In [15]:
#Model evaluation 
svd_results = evaluate_recommendations(svd_recs, test_relevant_sample, k=10)
print("SVD Model Results:")
for metric, value in svd_results.items():
    if metric != 'n_users_eval':
        print(f"  {metric:<15s}: {value:.4f}")
print(f"  Users evaluated: {svd_results['n_users_eval']:,}")

Evaluating:   0%|          | 0/2000 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 2000/2000 [00:00<00:00, 46045.97it/s]

SVD Model Results:
  NDCG@10        : 0.0669
  Precision@10   : 0.0607
  Recall@10      : 0.0221
  Users evaluated: 2,000


In [16]:
### Result comparison table 
print("-" * 80)
print("Baseline comparision Results")
print('-' * 80)
print()

models = {
    "Random": random_results,
    "Popularity": popularity_results,
    "SVD" : svd_results
}

print(f"{'Model':<15s} {'NDCG@10':>10s} {'Precision@10':>14s} {'Recall@10':>11s}")
print("-" * 55)

for model_name, results in models.items():
    ndcg  = results.get('NDCG@10', 0)
    prec  = results.get('Precision@10', 0)
    rec   = results.get('Recall@10', 0)
    print(f"{model_name:<15s} {ndcg:>10.4f} "
          f"{prec:>14.4f} {rec:>11.4f}")
    
print()
print("Key:")
print("  NDCG@10      → primary metric, ranking quality")
print("  Precision@10 → fraction of top-10 that are relevant")
print("  Recall@10    → fraction of relevant items in top-10")
print()

# Improvement over random
svd_ndcg    = svd_results['NDCG@10']
random_ndcg = random_results['NDCG@10']
pop_ndcg    = popularity_results['NDCG@10']

print(f"SVD vs Random     : "
      f"{100*(svd_ndcg - random_ndcg)/random_ndcg:.1f}% NDCG improvement")
print(f"SVD vs Popularity : "
      f"{100*(svd_ndcg - pop_ndcg)/pop_ndcg:.1f}% NDCG improvement")
print()
print("Target for NCF Two-Tower model:")
print(f"  Must beat SVD NDCG@10 of {svd_ndcg:.4f}")
print(f"  Target    : > {svd_ndcg * 1.15:.4f}  (15% improvement)")



--------------------------------------------------------------------------------
Baseline comparision Results
--------------------------------------------------------------------------------

Model              NDCG@10   Precision@10   Recall@10
-------------------------------------------------------
Random              0.0019         0.0018      0.0004
Popularity          0.0845         0.0749      0.0225
SVD                 0.0669         0.0607      0.0221

Key:
  NDCG@10      → primary metric, ranking quality
  Precision@10 → fraction of top-10 that are relevant
  Recall@10    → fraction of relevant items in top-10

SVD vs Random     : 3450.6% NDCG improvement
SVD vs Popularity : -20.9% NDCG improvement

Target for NCF Two-Tower model:
  Must beat SVD NDCG@10 of 0.0669
  Target    : > 0.0769  (15% improvement)


In [17]:
#Save model
import json

results_summary = {
    "model"       : "SVD_baseline",
    "n_factors"   : 100,
    "n_epochs"    : 20,
    "sample_size" : SAMPLE_SIZE,
    "metrics"     : {
        "random"     : random_results,
        "popularity" : popularity_results,
        "svd"        : svd_results,
    },
    "timestamp"   : datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

results_path = os.path.join(MODELS_DIR, "svd_baseline_results.json")
with open(results_path, "w") as f:
    json.dump(results_summary, f, indent=2)

print(f"Results saved to {results_path}")

Results saved to c:\Projects\Cinemate V2\models\svd_baseline_results.json


In [18]:
#Save summary
import json

results_summary = {
    "model"       : "SVD_baseline",
    "n_factors"   : 100,
    "n_epochs"    : 20,
    "sample_size" : SAMPLE_SIZE,
    "metrics"     : {
        "random"     : random_results,
        "popularity" : popularity_results,
        "svd"        : svd_results,
    },
    "timestamp"   : datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

results_path = os.path.join(MODELS_DIR, "svd_baseline_results.json")
with open(results_path, "w") as f:
    json.dump(results_summary, f, indent=2)

print(f"Results saved to {results_path}")

Results saved to c:\Projects\Cinemate V2\models\svd_baseline_results.json


In [19]:
import pickle

svd_path = os.path.join(MODELS_DIR, "svd_baseline.pkl")
with open(svd_path, "wb") as f:
    pickle.dump(svd_model, f)

print(f"SVD model saved to {svd_path}")
print()
print("To load later:")
print("  with open('models/svd_baseline.pkl', 'rb') as f:")
print("      svd = pickle.load(f)")
print("  pred = svd.predict(user_idx, movie_idx).est")

SVD model saved to c:\Projects\Cinemate V2\models\svd_baseline.pkl

To load later:
  with open('models/svd_baseline.pkl', 'rb') as f:
      svd = pickle.load(f)
  pred = svd.predict(user_idx, movie_idx).est


In [20]:
# Which users does SVD struggle with most?
# Understanding failures guides NCF design

user_ndcg_scores = {}

for user_idx in tqdm(eval_samples[:500], desc="Per-user analysis"):
    relevant  = test_relevant_sample.get(user_idx, set())
    if not relevant:
        continue
    recommended = svd_recs.get(user_idx, [])
    score = ndcg_at_k(recommended, relevant, k=10)
    user_ndcg_scores[user_idx] = score

scores_array = np.array(list(user_ndcg_scores.values()))

print("SVD Per-User NDCG@10 Distribution:")
print(f"  Mean   : {scores_array.mean():.4f}")
print(f"  Median : {np.median(scores_array):.4f}")
print(f"  Std    : {scores_array.std():.4f}")
print(f"  Min    : {scores_array.min():.4f}")
print(f"  Max    : {scores_array.max():.4f}")
print()

# Users with zero NDCG
zero_ndcg = (scores_array == 0).sum()
print(f"Users with NDCG = 0 : {zero_ndcg} "
      f"({100*zero_ndcg/len(scores_array):.1f}%)")
print()
print("Insight: Users with NDCG=0 are where NCF + content")
print("tower should show the most improvement over SVD.")

Per-user analysis: 100%|██████████| 500/500 [00:00<00:00, 28923.44it/s]

SVD Per-User NDCG@10 Distribution:
  Mean   : 0.0589
  Median : 0.0000
  Std    : 0.1161
  Min    : 0.0000
  Max    : 0.6875

Users with NDCG = 0 : 340 (68.0%)

Insight: Users with NDCG=0 are where NCF + content
tower should show the most improvement over SVD.


In [21]:
print("=" * 55)
print("KNOWN LIMITATIONS OF SVD BASELINE")
print("=" * 55)
print()
print("1. Cold Start — New Movies")
print("   SVD has no embedding for movies not in training set.")
print("   Our content tower solves this via title+genre+tags.")
print()
print("2. Cold Start — New Users")
print("   SVD cannot recommend to users with no history.")
print("   Our content tower + genome scores helps here.")
print()
print("3. Inference Speed")
print("   SVD scores every (user, movie) pair individually.")
print(f"   For {NUM_MOVIES:,} movies: too slow for real-time serving.")
print("   ChromaDB ANN search in Two-Tower is O(log n).")
print()
print("4. Non-Linear Patterns")
print("   SVD is linear — cannot capture complex interactions.")
print("   NCF MLP layers learn non-linear user-item patterns.")
print()
print("5. No Content Features")
print("   SVD uses ONLY rating history.")
print("   Two-Tower uses title + genres + genome tags as well.")

KNOWN LIMITATIONS OF SVD BASELINE

1. Cold Start — New Movies
   SVD has no embedding for movies not in training set.
   Our content tower solves this via title+genre+tags.

2. Cold Start — New Users
   SVD cannot recommend to users with no history.
   Our content tower + genome scores helps here.

3. Inference Speed
   SVD scores every (user, movie) pair individually.
   For 27,766 movies: too slow for real-time serving.
   ChromaDB ANN search in Two-Tower is O(log n).

4. Non-Linear Patterns
   SVD is linear — cannot capture complex interactions.
   NCF MLP layers learn non-linear user-item patterns.

5. No Content Features
   SVD uses ONLY rating history.
   Two-Tower uses title + genres + genome tags as well.
